# In-context inference of grid transformations — k = 2, `openai/gpt-6-luna` via OpenRouter

The model sees 3 input/output pairs of 6×6 digit grids, all produced by the same hidden transformation, and must
apply it to a new grid. The transformations are the 6 ordered pairs of distinct operations: AB, AC, BA, BC, CA, CB
(A = shift every row right by one with wrap-around, B = rotate 90° clockwise, C = mirror left–right; left to right).
The prompt lists the three operations but never says which ones are used.

**Two experiments on the same tasks.** They differ only in the final instruction:
* `answer`: give the output grid;
* `ops`: first state the transformation as a sequence of the operations (`<ops>…</ops>`), then give the grid.

**Task construction.** The demos are checked so that exactly one of the 288 transformations fits them. The query
grid has 288 distinct images, so every answer identifies the transformation it corresponds to (or none).

**Before running.**
1. Create an OpenRouter API key (openrouter.ai → Keys; set a credit limit).
2. In Colab, open the key icon (Secrets) on the left and add a secret named `OPENROUTER_API_KEY` with the key as its
   value; allow this notebook to access it. Never paste the key into a cell.

**Order.** Cell 4 runs a dry run (no API calls, no cost). Cell 5 runs a 4-call pilot: check the responses and
the cost. Cell 6 runs everything (resumable: calls already made are skipped). Cell 7 scores and downloads the
results. A CPU runtime is enough.


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- 3. Configuration ---------------------------------------------------------------------------------------------
MODEL = "openai/gpt-6-luna"
N_PER_WORD = 20                  # tasks per transformation (6 transformations -> 120 tasks, 240 calls for 2 experiments)
N_DEMOS = 3
TASK_SEED = 0
EXPERIMENTS = ["answer", "ops"]
REASONING = None                 # None = the model's default; e.g. {"effort": "high"} or {"max_tokens": 8000}
MAX_TOKENS = None                # cap on output tokens per call (None = provider default)
WORKERS = 4                      # parallel API calls
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if IN_COLAB and not os.environ.get("OPENROUTER_API_KEY"):
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")      # never printed
print("OpenRouter key present:", bool(os.environ.get("OPENROUTER_API_KEY")))
import json, pathlib
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 300); pd.set_option("display.width", 220)
from decompositionality import icl_api as IA
_tag = "default" if not REASONING else "_".join(f"{k}{v}" for k, v in REASONING.items())
OUT_DIR = pathlib.Path(RESULTS_ROOT) / "icl_api" / f"k2_{MODEL.replace('/', '_')}_demos{N_DEMOS}_seed{TASK_SEED}_{_tag}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSONL = OUT_DIR / "responses.jsonl"
print("results ->", OUT_DIR)


In [ ]:
# --- 4. Dry run: build the tasks and show one prompt per experiment (no API calls) ----------------------------------
tasks = IA.make_tasks(IA.K2_WORDS, n_per_word=N_PER_WORD, n_demos=N_DEMOS, seed=TASK_SEED)
(OUT_DIR / "tasks.json").write_text(json.dumps(tasks))
print(len(tasks), "tasks:", {w: sum(t["word"] == w for t in tasks) for w in IA.K2_WORDS})
for e in EXPERIMENTS:
    print(f"\n==================== experiment '{e}' (task {tasks[0]['task_id']}) ====================")
    print(IA.build_prompt(tasks[0], e))
print("\nExpected answer for", tasks[0]["task_id"], ":")
print(IA._fmt(tasks[0]["expected"]))


In [ ]:
# --- 5. Pilot: 4 calls, then inspect the raw responses and the cost ------------------------------------------------
IA.run(tasks, OUT_JSONL, MODEL, experiments=EXPERIMENTS, reasoning=REASONING, max_tokens=MAX_TOKENS, workers=WORKERS,
       limit=4)
for line in OUT_JSONL.read_text().splitlines()[-4:]:
    r = json.loads(line)
    print(f"\n----- {r['task_id']} / {r['experiment']} | model {r['model']} | {r['seconds']} s | error: {r['error']}")
    print((r["content"] or "")[:1500])
    print("usage:", r["usage"])
display(IA.score(tasks, OUT_JSONL))


In [ ]:
# --- 6. Full run (resumable: calls already in responses.jsonl are skipped) -------------------------------------------
IA.run(tasks, OUT_JSONL, MODEL, experiments=EXPERIMENTS, reasoning=REASONING, max_tokens=MAX_TOKENS, workers=WORKERS)


In [ ]:
# --- 7. Scores and summary --------------------------------------------------------------------------------------------
df = IA.score(tasks, OUT_JSONL)
S = IA.summarize(df)
for name, tab in S.items():
    print(f"\n### {name}")
    display(tab)
df.to_csv(OUT_DIR / "scores.csv", index=False)
for name, tab in S.items():
    tab.to_csv(OUT_DIR / f"summary_{name}.csv", index=False)
print("saved to", OUT_DIR)
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    files.download(str(OUT_DIR) + ".zip")
